# Spark: bank-loan data analysis

Run the cells in order. This workshop uses the existing bank-loan dataset
and Spark master/worker setup. Container management is handled by the organiser.

You will inspect the data, connect to your group's Spark cluster, filter records,
and save a small result in your own workspace. No participant installation is required
once workspace preparation has been validated.

## 1. Inspect your local dataset

The CSV is included under `data/`. Preview it before asking Spark to process it.
`Income` and other field meanings should be checked against the accompanying
teaching material; the CSV itself does not specify their units.

In [ ]:
import csv
from pathlib import Path

workspace = Path.cwd()
dataset = workspace / 'data' / 'Bank_Personal_Loan_Modelling.csv'
if not dataset.is_file():
    raise FileNotFoundError('Open this notebook from the Spark workspace directory.')

with dataset.open(newline='', encoding='utf-8-sig') as handle:
    reader = csv.DictReader(handle)
    print('Columns:', ', '.join(reader.fieldnames))
    for index, row in enumerate(reader):
        if index == 5:
            break
        print(row)


## 2. Connect to the prepared Spark cluster

The organiser supplies `SPARK_MASTER_URL`, `SPARK_DATA_PATH` and
`SPARK_DRIVER_HOST`. The dataset path must be accessible to the notebook driver
and Spark workers. The driver address must be reachable from the worker containers.

If the configuration check fails, stop and report it to the organiser.
Do not install software or start Docker containers yourself.

In [ ]:
import os

required_settings = ('SPARK_MASTER_URL', 'SPARK_DATA_PATH', 'SPARK_DRIVER_HOST')
missing = [name for name in required_settings if not os.environ.get(name)]
if missing:
    raise RuntimeError('Workspace preparation is incomplete: missing ' + ', '.join(missing))

from pyspark.sql import SparkSession

builder = (
    SparkSession.builder
    .appName('BankLoanAnalysis')
    .master(os.environ['SPARK_MASTER_URL'])
    .config('spark.driver.host', os.environ['SPARK_DRIVER_HOST'])
    .config('spark.driver.bindAddress', '0.0.0.0')
)
for setting, environment in (
    ('spark.driver.port', 'SPARK_DRIVER_PORT'),
    ('spark.blockManager.port', 'SPARK_BLOCK_MANAGER_PORT'),
):
    if os.environ.get(environment):
        builder = builder.config(setting, os.environ[environment])

spark = builder.getOrCreate()
print('Spark version:', spark.version)
print('Spark master:', spark.sparkContext.master)


## 3. Read the CSV and inspect its schema

`header` uses the first row as column names. `inferSchema` asks Spark to infer
column types from the data. These are the same options used by `data-analytics.py`.

In [ ]:
df = (
    spark.read.format('csv')
    .option('header', 'true')
    .option('inferSchema', 'true')
    .load(os.environ['SPARK_DATA_PATH'])
)
df.printSchema()
df.show(5)
print('Number of records:', df.count())


## 4. Filter customers older than 30

Select `Age`, `Income` and `Family`, then retain records where `Age > 30`.
`select` and `filter` construct a plan; `show` triggers its execution.

In [ ]:
result_df = df.select('Age', 'Income', 'Family').filter(df.Age > 30)
result_df.show(10)
print('Matching records:', result_df.count())


## 5. Explore the loan outcome

Count records for each value of `Personal Loan`. A Spark aggregation computes
these counts across the input partitions.

After running the cell, identify which category contains more records.

In [ ]:
df.groupBy('Personal Loan').count().orderBy('Personal Loan').show()


## 6. Save a small preview in your own workspace

Collect only ten rows, then write them locally. This avoids collecting the entire
dataset and avoids writing participant results into a shared container directory.
The selected ten rows are not guaranteed to have a fixed order.

In [ ]:
output_dir = workspace / 'outputs'
output_dir.mkdir(exist_ok=True)
output_file = output_dir / 'filtered_preview.csv'

with output_file.open('w', newline='', encoding='utf-8') as handle:
    writer = csv.writer(handle)
    writer.writerow(result_df.columns)
    for row in result_df.limit(10).collect():
        writer.writerow(tuple(row))

print('Saved:', output_file)


## 7. Finish

Run the final cell when you have finished. To repeat the Spark exercises afterwards,
restart the notebook kernel and run the notebook from the beginning.

For additional background, open `docs/spark-DA.pdf`. The original container tutorial
is retained as `participants-instructions.md`; its Docker commands describe the
standalone workshop, not participant setup in JupyterHub.

In [ ]:
spark.stop()
print('Spark session stopped.')
